# Lesson 11: A clean package + automatic tests

**Why:** until now, every notebook re-typed the whole model from scratch. That's fine for learning, but in real projects code lives in **one place** and gets **imported**. Then every notebook and script uses the same, tested version.

The GPT from Lesson 10 now lives in proper files:

```
src/tinygpt/
├─ tokenizer.py   CharTokenizer        (Lesson 4)
├─ data.py        load_text, split_data, get_batch   (Lesson 5)
└─ model.py       GPTConfig, Head, MultiHeadAttention, FeedForward, Block, GPT   (Lessons 8–10)
tests/
└─ test_gpt.py    8 automatic checks
```

It's the **same code** as Lesson 10, with one change: all the size settings now live together in a **`GPTConfig`**, so making the model bigger or smaller means changing one line.

Open [`src/tinygpt/model.py`](../src/tinygpt/model.py) side by side with this notebook if you like. You'll recognize every class.

## 1. Using the package

Three imports replace about 100 lines of copy-pasted code.

In [ ]:
import math
import subprocess
import sys
import torch

from tinygpt.data import load_text, split_data
from tinygpt.model import GPT, GPTConfig
from tinygpt.tokenizer import CharTokenizer

torch.manual_seed(1337)

text = load_text()
tok = CharTokenizer.from_text(text)
train_data, val_data = split_data(tok.encode(text))

config = GPTConfig(vocab_size=tok.vocab_size)
model = GPT(config)
print(config)
print(f"parameters: {model.num_params():,}")

## 2. Counting the parameters by hand

Where do the 214,357 numbers live? With C = 64, V = 85, T = 64:

| Part | Formula | Count |
|---|---|---|
| Token embeddings | V × C | |
| Position embeddings | T × C | |
| Each block: 2 LayerNorms | 2 × 2C | |
| Each block: query + key + value | 3 × C × C | |
| Each block: projection | C × C + C | |
| Each block: feed-forward | C × 4C + 4C + 4C × C + C | |
| Final LayerNorm | 2C | |
| Output layer | C × V + V | |

(A `Linear` layer from *a* numbers to *b* numbers has a × b weights + b biases. A LayerNorm has 2 knobs per number.)

In [ ]:
C, V, T, L = config.n_embd, config.vocab_size, config.block_size, config.n_layer

parts = {
    "token embeddings":      V * C,
    "position embeddings":   T * C,
    "blocks: LayerNorms":    L * (2 * 2 * C),
    "blocks: q, k, v":       L * (3 * C * C),
    "blocks: projection":    L * (C * C + C),
    "blocks: feed-forward":  L * (C * 4 * C + 4 * C + 4 * C * C + C),
    "final LayerNorm":       2 * C,
    "output layer":          C * V + V,
}
for name, count in parts.items():
    print(f"{name:22} {count:>8,}   {count / model.num_params():5.1%}")
print(f"{'TOTAL by hand':22} {sum(parts.values()):>8,}")
print(f"{'TOTAL by PyTorch':22} {model.num_params():>8,}")

## 3. The most important test: no peeking at the future

The mask from Lesson 7 is a single line of code. If it breaks, **training still runs and the loss even looks great**, because the model cheats by reading the next character. It would then write garbage when generating, since the future doesn't exist yet. So we test it directly:

1. Run the model on some text.
2. Change **only the character at position 5**, and run again.
3. Predictions at positions **0–4** must be **exactly the same**. They're not allowed to know about position 5.

**Predict:** first we test the healthy model. Then we **break the mask on purpose** (every head may see everything). With the broken mask, will changing position 5 alter the predictions at positions 0–4?

In [ ]:
def peeking_amount(model):
    # How much do predictions at positions 0-4 change when we only change position 5?
    model.eval()
    idx = torch.tensor([tok.encode("Tesla's alternating current")[: config.block_size]])
    changed = idx.clone()
    changed[0, 5] = tok.stoi["X"]
    with torch.no_grad():
        before, _ = model(idx)
        after, _ = model(changed)
    return (before[:, :5] - after[:, :5]).abs().max().item()

print(f"healthy model: positions 0-4 changed by {peeking_amount(model):.6f}")

broken = GPT(config)
broken.load_state_dict(model.state_dict())   # identical copy...
for block in broken.blocks:
    for head in block.attn.heads:
        head.tril.fill_(1)                    # ...but every head may now see the future
print(f"broken mask:   positions 0-4 changed by {peeking_amount(broken):.6f}")

## 4. Run the whole test suite

[`tests/test_gpt.py`](../tests/test_gpt.py) holds 8 checks. Each one guards a lesson:

| Test | Guards |
|---|---|
| tokenizer round trip | Lesson 4: nothing lost; unknown letters (ć) are rejected |
| batch targets shifted by one | Lesson 5 |
| forward shapes | (B, T) in → (B, T, V) out |
| initial loss ≈ ln(85) | Lesson 3: a fresh model knows nothing, and isn't confidently wrong |
| no peeking at the future | Lesson 7's mask |
| can memorize one batch | the model *can* learn: given one batch 300 times, its loss must go almost to 0 |
| parameter count = formula | section 2 above |
| generate | writes the right number of characters, even from a long starting text |

The tests run in **pytest**, the standard Python testing tool. From a terminal you'd type `uv run pytest`. Here we start it from the notebook (it takes about 15 seconds).

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-v", "tests"],
                        cwd="..", capture_output=True, text=True)
print(result.stdout[-1500:])

## 5. Final check

In [ ]:
assert sum(parts.values()) == model.num_params()
assert peeking_amount(model) == 0.0
assert peeking_amount(broken) > 0.0
assert result.returncode == 0, "some tests failed"
print("Lesson 11 complete. The GPT is now a tested package.")

## Wrap-up question

The model trains fine whether the mask works or not. So why is it worth having an automatic test that proves "no peeking"? What would you see if the mask were broken and nobody noticed?